# AuxoYeast reproducibility audit

This notebook is an analysis interface for the `auxoyeast_repro` package. Core simulation logic lives under `src/` and is not duplicated here.


In [ ]:
from pathlib import Path
import pandas as pd

from auxoyeast_repro import AuditConfig, ProjectPaths, load_model, model_qc, parse_dataset, run_benchmark
from auxoyeast_repro.reporting import compare_pair_results, summarize_results

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PATHS = ProjectPaths(ROOT)
CONFIG = AuditConfig(solver="glpk", isolation_mode="reload")


## Model quality control


In [ ]:
original = load_model(PATHS.original_xml, CONFIG.solver)
curated = load_model(PATHS.curated_xml, CONFIG.solver)
qc = pd.DataFrame([
    model_qc(original, "Yeast9"),
    model_qc(curated, "Yeast9_curated"),
])
qc


## Dataset validation


In [ ]:
pairs = parse_dataset(PATHS.dataset_xlsx)
assert len(pairs) == 147
pairs.head()


## Reference threshold


In [ ]:
wt_original = float(qc.loc[qc["model"] == "Yeast9", "wt_growth"].iloc[0])
threshold = CONFIG.viability_fraction * wt_original
threshold


## Strict isolated benchmark


In [ ]:
ALIASES = {
    "Yeast9": {},
    "Yeast9_curated": {"a_0001": "r_temp1"},
}

original_results = run_benchmark(
    model_path=PATHS.original_xml,
    dataset_path=PATHS.dataset_xlsx,
    model_label="Yeast9",
    pairs_df=pairs,
    threshold=threshold,
    aliases=ALIASES["Yeast9"],
    output_dir=PATHS.output_dir,
    config=CONFIG,
    resume=True,
)


In [ ]:
curated_results = run_benchmark(
    model_path=PATHS.curated_xml,
    dataset_path=PATHS.dataset_xlsx,
    model_label="Yeast9_curated",
    pairs_df=pairs,
    threshold=threshold,
    aliases=ALIASES["Yeast9_curated"],
    output_dir=PATHS.output_dir,
    config=CONFIG,
    resume=True,
)


## Summary


In [ ]:
summary = pd.DataFrame([
    {"model": "Yeast9", **summarize_results(original_results, threshold)},
    {"model": "Yeast9_curated", **summarize_results(curated_results, threshold)},
])
summary["accuracy"] = summary["correct"] / summary["n"]
summary


## Pairwise curation effect


In [ ]:
pairwise = compare_pair_results(original_results, curated_results)
pairwise["change"].value_counts()


In [ ]:
pairwise.loc[
    pairwise["change"].isin(["fixed", "regression"]),
    [
        "pair_id",
        "gene_field_original",
        "chemical_original",
        "classification_original",
        "classification_curated",
        "change",
    ],
]
